In [1]:
# ═══════════════════════════════════════════════════════════
#  COMPONENT 1 — PER-JUNCTION TIME-BASED TRAIN / VAL SPLIT
#  Each junction is split independently on the same month
#  boundary; train rows and val rows are labelled by junction.
# ═══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os

SAVE_DIR  = r"C:\Users\syin\Downloads\Uber_project\Datasets"
DATA_PATH = os.path.join(SAVE_DIR, "engineered_ML_traffic_data.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset shape : {df.shape}")

# ── Feature columns ──────────────────────────────────────
TARGET = "Vehicles"

TEMPORAL = ["hour","hour_sin","hour_cos","day_of_week",
            "day_of_week_sin","day_of_week_cos","month","year"]
JUNCTION  = ["junction"]
CALENDAR  = ["is_weekend","is_workday","is_rush_hour",
             "is_holiday","is_special_event"]
WEATHER   = ["temp_norm","wspd_norm","pres","pres_norm"]
TIME_ENC  = ["time_of_day_enc"]
LAG       = ["lag_1h","lag_2h","lag_3h","lag_6h",
             "lag_12h","lag_24h","lag_168h"]
ROLLING   = ["rolling_mean_3h","rolling_mean_6h","rolling_mean_24h",
             "rolling_std_3h","rolling_std_6h","rolling_std_24h",
             "rolling_max_3h","rolling_max_6h","rolling_max_24h",
             "rolling_min_3h","rolling_min_6h","rolling_min_24h",
             "ewma_12h","ewma_24h"]

ALL_FEATURES = TEMPORAL + JUNCTION + CALENDAR + WEATHER + TIME_ENC + LAG + ROLLING
FEATURE_COLS = [c for c in ALL_FEATURES if c in df.columns]

missing = [c for c in ALL_FEATURES if c not in df.columns]
if missing:
    print(f"\n⚠  Missing from dataset : {missing}")

non_num = df[FEATURE_COLS].select_dtypes(include="object").columns.tolist()
if non_num:
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in non_num]

print(f"✔ Feature set : {len(FEATURE_COLS)} columns")

# ════════════════════════════════════════════════════════
#  STEP 1 — PER-JUNCTION TIME-BASED SPLIT
#
#  Split boundary is the same calendar cut for every junction:
#    Train : year < 2017  OR  (year == 2017 AND month < 5)
#    Val   : year == 2017 AND month >= 5
#
#  The split is applied within each junction independently,
#  so junction 1 trains only on junction-1 rows and validates
#  only on junction-1 rows — no cross-junction leakage.
# ════════════════════════════════════════════════════════

SPLIT_YEAR  = 2017
SPLIT_MONTH = 5          # May 2017 and later → validation

junctions = sorted(df["junction"].unique())
print(f"\nJunctions found : {junctions}")

# ── Apply split per junction and collect into train / val ─
train_parts = []
val_parts   = []
split_report = []

for jn in junctions:
    jdf = df[df["junction"] == jn].copy()

    train_mask = (jdf["year"] < SPLIT_YEAR) | \
                 ((jdf["year"] == SPLIT_YEAR) & (jdf["month"] < SPLIT_MONTH))
    val_mask   = (jdf["year"] == SPLIT_YEAR) & (jdf["month"] >= SPLIT_MONTH)

    jn_train = jdf[train_mask].copy()
    jn_val   = jdf[val_mask].copy()

    # NaN fill within each junction split (no leakage between junctions)
    for split_part, name in [(jn_train, "train"), (jn_val, "val")]:
        n_nan = split_part[FEATURE_COLS].isna().sum().sum()
        if n_nan > 0:
            split_part[FEATURE_COLS] = (split_part[FEATURE_COLS]
                                        .bfill().ffill())

    train_parts.append(jn_train)
    val_parts.append(jn_val)

    split_report.append({
        "junction" : jn,
        "train_rows": len(jn_train),
        "val_rows"  : len(jn_val),
        "train_pct" : round(100 * len(jn_train) / max(len(jdf), 1), 1),
        "val_pct"   : round(100 * len(jn_val)   / max(len(jdf), 1), 1),
        "train_months": (f"{jn_train['year'].min()}-{jn_train['month'].min():02d}"
                         f" → {jn_train['year'].max()}-{jn_train['month'].max():02d}"
                         if len(jn_train) else "—"),
        "val_months"  : (f"{jn_val['year'].min()}-{jn_val['month'].min():02d}"
                         f" → {jn_val['year'].max()}-{jn_val['month'].max():02d}"
                         if len(jn_val) else "—"),
    })

train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts,   ignore_index=True)

# ── Report ────────────────────────────────────────────────
print("\n── Per-Junction Split Report ─────────────────────────────")
report_df = pd.DataFrame(split_report)
print(report_df.to_string(index=False))
print(f"\n  Total train rows : {len(train_df)}")
print(f"  Total val rows   : {len(val_df)}")
print(f"  Overall val %    : {100 * len(val_df) / len(df):.1f}%")

# Remaining NaN check
remaining = (train_df[FEATURE_COLS].isna().sum().sum() +
             val_df[FEATURE_COLS].isna().sum().sum())
print(f"\n{'✅' if remaining == 0 else '⚠ '} Remaining NaN in features: {remaining}")

# ── X / y ─────────────────────────────────────────────────
X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_val   = val_df[FEATURE_COLS]
y_val   = val_df[TARGET]

print(f"\nX_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   y_val   : {y_val.shape}")

# ── Save ──────────────────────────────────────────────────
train_df.to_csv(os.path.join(SAVE_DIR, "train_split.csv"), index=False)
val_df.to_csv(  os.path.join(SAVE_DIR, "val_split.csv"),   index=False)
pd.Series(FEATURE_COLS).to_csv(
    os.path.join(SAVE_DIR, "feature_cols.csv"),
    index=False, header=["feature"]
)
report_df.to_csv(os.path.join(SAVE_DIR, "split_report.csv"), index=False)

print("\n✔ train_split.csv saved.")
print("✔ val_split.csv saved.")
print("✔ feature_cols.csv saved.")
print("✔ split_report.csv saved.")
print("\n✅ Component 1 complete — per-junction split ready.")

Loaded dataset shape : (48120, 41)
✔ Feature set : 40 columns

Junctions found : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

── Per-Junction Split Report ─────────────────────────────
 junction  train_rows  val_rows  train_pct  val_pct      train_months        val_months
        1       12552      2040       86.0     14.0 2015-01 → 2017-12 2017-05 → 2017-12
        2       12552      2040       86.0     14.0 2015-01 → 2017-12 2017-05 → 2017-12
        3       12552      2040       86.0     14.0 2015-01 → 2017-12 2017-05 → 2017-12
        4        2304      2040       53.0     47.0 2017-01 → 2017-04 2017-05 → 2017-12

  Total train rows : 39960
  Total val rows   : 8160
  Overall val %    : 17.0%

✅ Remaining NaN in features: 0

X_train : (39960, 40)   y_train : (39960,)
X_val   : (8160, 40)   y_val   : (8160,)

✔ train_split.csv saved.
✔ val_split.csv saved.
✔ feature_cols.csv saved.
✔ split_report.csv saved.

✅ Component 1 complete — per-junction split ready.


In [5]:
# ═══════════════════════════════════════════════════════════
#  COMPONENT 2 — MODEL TRAINING  (per-junction strategy)
#
#  Two training modes — choose via TRAIN_MODE:
#
#  "global"   : One model trained on ALL junction rows
#               → validated per junction (recommended first)
#
#  "per_jn"   : One separate model per junction
#               → each model trained/validated on its own rows
#               (better accuracy, more models to store)
# ═══════════════════════════════════════════════════════════
!pip install xgboost lightgbm
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "xgboost", "lightgbm"], check=True)

# Force-reload without restarting kernel
import importlib
import xgboost as xgb
import lightgbm as lgb
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)
import pandas as pd
import numpy as np
import os, warnings, joblib
warnings.filterwarnings("ignore")

from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
import xgboost  as xgb
import lightgbm as lgb

TRAIN_MODE = "global"   # "global"  or  "per_jn"

SAVE_DIR  = r"C:\Users\syin\Downloads\Uber_project\Datasets"
MODEL_DIR = os.path.join(SAVE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(SAVE_DIR, "train_split.csv"))
val_df   = pd.read_csv(os.path.join(SAVE_DIR, "val_split.csv"))
FEATURE_COLS = pd.read_csv(
    os.path.join(SAVE_DIR, "feature_cols.csv"))["feature"].tolist()
TARGET    = "Vehicles"
junctions = sorted(train_df["junction"].unique())
tscv      = TimeSeriesSplit(n_splits=5)

print(f"Training mode : {TRAIN_MODE}")
print(f"Junctions     : {junctions}\n")

# ── Metric helper ─────────────────────────────────────────
def metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) /
                           np.where(y_true == 0, 1, y_true))) * 100
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

# ── Model definitions (shared across modes) ───────────────
def make_models():
    xgb_param = {
        "n_estimators":[200,400,600], "max_depth":[4,6,8],
        "learning_rate":[0.01,0.05,0.1], "subsample":[0.7,0.9,1.0],
        "colsample_bytree":[0.7,0.9,1.0], "min_child_weight":[1,3,5],
        "gamma":[0,0.1,0.3],
    }
    lgb_param = {
        "n_estimators":[200,400,600], "max_depth":[4,6,8,-1],
        "learning_rate":[0.01,0.05,0.1], "num_leaves":[31,63,127],
        "subsample":[0.7,0.9,1.0], "colsample_bytree":[0.7,0.9,1.0],
        "min_child_samples":[10,20,50], "reg_alpha":[0,0.1,0.5],
        "reg_lambda":[0,0.1,1.0],
    }
    rf_param = {
        "n_estimators":[100,200,400], "max_depth":[None,10,20,30],
        "max_features":["sqrt","log2",0.5], "min_samples_split":[2,5,10],
        "min_samples_leaf":[1,2,4], "bootstrap":[True,False],
    }
    return {
        "XGBoost": (
            xgb.XGBRegressor(objective="reg:squarederror",
                             random_state=42, n_jobs=-1, verbosity=0),
            xgb_param, 30
        ),
        "LightGBM": (
            lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
            lgb_param, 30
        ),
        "RandomForest": (
            RandomForestRegressor(random_state=42, n_jobs=-1),
            rf_param, 20
        ),
    }

def fit_model(name, base_est, param_dist, n_iter, X_tr, y_tr):
    search = RandomizedSearchCV(
        base_est, param_dist, n_iter=n_iter, cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1, random_state=42, verbose=0
    )
    search.fit(X_tr, y_tr)
    return search.best_estimator_, search.best_params_

# ════════════════════════════════════════════════════════
#  MODE A — GLOBAL MODEL (one model, validated per junction)
# ════════════════════════════════════════════════════════
if TRAIN_MODE == "global":
    X_tr = train_df[FEATURE_COLS].values
    y_tr = train_df[TARGET].values

    all_rows = []
    for mname, (base_est, param_dist, n_iter) in make_models().items():
        print(f"── Training {mname} (global) ─────────────────────")
        model, best_p = fit_model(mname, base_est, param_dist, n_iter,
                                  X_tr, y_tr)
        print(f"  Best params: {best_p}")
        joblib.dump(model, os.path.join(MODEL_DIR, f"{mname.lower()}_global.pkl"))

        # ── Validate per junction ─────────────────────────
        print(f"  Per-junction validation:")
        for jn in junctions:
            mask = val_df["junction"] == jn
            X_vj = val_df.loc[mask, FEATURE_COLS].values
            y_vj = val_df.loc[mask, TARGET].values
            if len(y_vj) == 0:
                continue
            preds = model.predict(X_vj)
            m = metrics(y_vj, preds)
            print(f"    Jn {jn} → MAE={m['MAE']:.4f}  "
                  f"RMSE={m['RMSE']:.4f}  R²={m['R2']:.4f}")
            all_rows.append({"Model": mname, "Junction": jn, **m,
                             "preds": preds, "mask_idx": val_df[mask].index})

    # Assemble predictions DataFrame
    preds_df = val_df[["junction","year","month","hour",TARGET]].copy()
    for mname in make_models():
        preds_df[f"pred_{mname}"] = np.nan
    for row in all_rows:
        preds_df.loc[row["mask_idx"], f"pred_{row['Model']}"] = row["preds"]

    preds_df.to_csv(os.path.join(MODEL_DIR, "val_predictions.csv"), index=False)

    # Summary
    summary_rows = []
    for row in all_rows:
        summary_rows.append({
            "Model": row["Model"], "Junction": row["Junction"],
            "MAE": round(row["MAE"],4), "RMSE": round(row["RMSE"],4),
            "R2": round(row["R2"],4), "MAPE(%)": round(row["MAPE"],4)
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(MODEL_DIR, "model_comparison.csv"), index=False)
    print("\n══ GLOBAL MODEL — VALIDATION SUMMARY (per junction) ══")
    print(summary_df.to_string(index=False))

# ════════════════════════════════════════════════════════
#  MODE B — PER-JUNCTION MODELS
# ════════════════════════════════════════════════════════
elif TRAIN_MODE == "per_jn":
    preds_df = val_df[["junction","year","month","hour",TARGET]].copy()
    all_summary = []

    for jn in junctions:
        print(f"\n══ Junction {jn} ══════════════════════════════════")
        tr_jn = train_df[train_df["junction"] == jn]
        vl_jn = val_df[val_df["junction"] == jn]
        X_tr  = tr_jn[FEATURE_COLS].values
        y_tr  = tr_jn[TARGET].values
        X_vl  = vl_jn[FEATURE_COLS].values
        y_vl  = vl_jn[TARGET].values

        if len(y_vl) == 0:
            print(f"  ⚠ No val rows for junction {jn} — skipping.")
            continue

        jn_preds = {}  # model_name → predictions on val rows

        for mname, (base_est, param_dist, n_iter) in make_models().items():
            print(f"  ── {mname} ──")
            model, best_p = fit_model(mname, base_est, param_dist,
                                      n_iter, X_tr, y_tr)
            preds = model.predict(X_vl)
            m     = metrics(y_vl, preds)
            print(f"    Best  : {best_p}")
            print(f"    MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  "
                  f"R²={m['R2']:.4f}  MAPE={m['MAPE']:.2f}%")

            joblib.dump(model,
                os.path.join(MODEL_DIR,
                             f"{mname.lower()}_junction{jn}.pkl"))
            jn_preds[mname] = preds
            all_summary.append({
                "Junction": jn, "Model": mname,
                "MAE": round(m["MAE"],4), "RMSE": round(m["RMSE"],4),
                "R2": round(m["R2"],4), "MAPE(%)": round(m["MAPE"],4)
            })

        # Write per-junction predictions into combined df
        for mname, p in jn_preds.items():
            col = f"pred_{mname}"
            if col not in preds_df.columns:
                preds_df[col] = np.nan
            preds_df.loc[vl_jn.index, col] = p

    preds_df.to_csv(os.path.join(MODEL_DIR, "val_predictions.csv"), index=False)
    summary_df = pd.DataFrame(all_summary)
    summary_df.to_csv(os.path.join(MODEL_DIR, "model_comparison.csv"),
                      index=False)
    print("\n══ PER-JUNCTION MODEL SUMMARY ══════════════════════")
    print(summary_df.sort_values(["Junction","MAE"]).to_string(index=False))

print("\n✔ val_predictions.csv saved.")
print("✔ model_comparison.csv saved.")
print("\n✅ Component 2 complete.")

xgboost: 3.2.0
lightgbm: 4.6.0
Training mode : global
Junctions     : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

── Training XGBoost (global) ─────────────────────
  Best params: {'subsample': 0.9, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}
  Per-junction validation:
    Jn 1 → MAE=4.7145  RMSE=6.4318  R²=0.9277
    Jn 2 → MAE=2.6755  RMSE=3.4455  R²=0.7207
    Jn 3 → MAE=4.3623  RMSE=5.5131  R²=0.4928
    Jn 4 → MAE=2.0666  RMSE=2.6107  R²=0.3557
── Training LightGBM (global) ─────────────────────
  Best params: {'subsample': 1.0, 'reg_lambda': 0, 'reg_alpha': 0, 'num_leaves': 31, 'n_estimators': 400, 'min_child_samples': 10, 'max_depth': -1, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
  Per-junction validation:
    Jn 1 → MAE=4.3948  RMSE=6.1314  R²=0.9343
    Jn 2 → MAE=2.5038  RMSE=3.1945  R²=0.7599
    Jn 3 → MAE=4.3404  RMSE=5.5399  R²=0.4879
    Jn 4 → MAE=2.0756  RMSE=2.6224  R²=0.3499


In [6]:
# ═══════════════════════════════════════════════════════════
#  COMPONENT 3 — EVALUATION & CROSS-VALIDATION
#  All metrics are computed per junction then aggregated.
# ═══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os, warnings, joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

SAVE_DIR  = r"C:\Users\syin\Downloads\Uber_project\Datasets"
MODEL_DIR = os.path.join(SAVE_DIR, "models")
PLOT_DIR  = os.path.join(SAVE_DIR, "evaluation_plots")
os.makedirs(PLOT_DIR, exist_ok=True)

TARGET = "Vehicles"

train_df = pd.read_csv(os.path.join(SAVE_DIR, "train_split.csv"))
val_df   = pd.read_csv(os.path.join(SAVE_DIR, "val_split.csv"))
preds_df = pd.read_csv(os.path.join(MODEL_DIR, "val_predictions.csv"))
FEATURE_COLS = pd.read_csv(
    os.path.join(SAVE_DIR, "feature_cols.csv"))["feature"].tolist()

junctions  = sorted(val_df["junction"].unique())
PRED_COLS  = [c for c in preds_df.columns if c.startswith("pred_")]
MODEL_NAMES= [c.replace("pred_","") for c in PRED_COLS]

print(f"Models    : {MODEL_NAMES}")
print(f"Junctions : {junctions}\n")

# ── Metric helper ─────────────────────────────────────────
def calc_metrics(y, p, model, junction):
    mae  = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    r2   = r2_score(y, p)
    mape = np.mean(np.abs((y - p) / np.where(y == 0, 1, y))) * 100
    return {"Model": model, "Junction": junction,
            "MAE": round(mae,4), "RMSE": round(rmse,4),
            "R2": round(r2,4), "MAPE(%)": round(mape,4)}

# ════════════════════════════════════════════════════════
#  3A.  PER-JUNCTION METRICS TABLE
# ════════════════════════════════════════════════════════
print("── 3A. Per-Junction Validation Metrics ─────────────")
metric_rows = []

for col, mname in zip(PRED_COLS, MODEL_NAMES):
    print(f"\n  Model : {mname}")
    for jn in junctions:
        mask = preds_df["junction"] == jn
        y    = preds_df.loc[mask, TARGET].values
        p    = preds_df.loc[mask, col].dropna().values
        if len(p) == 0 or len(y) == 0:
            continue
        # Align lengths if predictions are partial
        n = min(len(y), len(p))
        row = calc_metrics(y[:n], p[:n], mname, jn)
        metric_rows.append(row)
        print(f"    Jn {jn} → MAE={row['MAE']}  RMSE={row['RMSE']}  "
              f"R²={row['R2']}  MAPE={row['MAPE(%)']}%")

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(MODEL_DIR, "evaluation_metrics.csv"), index=False)
print(f"\n✔ evaluation_metrics.csv saved.")

# ── Overall (mean across junctions) per model ────────────
print("\n── Overall (mean across junctions) ──────────────────")
overall = (metrics_df.groupby("Model")[["MAE","RMSE","R2","MAPE(%)"]]
           .mean().round(4).sort_values("MAE"))
print(overall.to_string())
best_model_name = overall.index[0]
print(f"\n🏆 Best model (mean MAE) : {best_model_name}  "
      f"MAE={overall.loc[best_model_name,'MAE']}")

# ════════════════════════════════════════════════════════
#  3B.  PER-JUNCTION PLOTS (one figure per model)
# ════════════════════════════════════════════════════════
print("\n── 3B. Generating per-junction evaluation plots ─────")

for col, mname in zip(PRED_COLS, MODEL_NAMES):
    fig, axes = plt.subplots(len(junctions), 3,
                             figsize=(20, 5 * len(junctions)))
    if len(junctions) == 1:
        axes = [axes]           # keep indexing consistent
    fig.suptitle(f"Per-Junction Evaluation — {mname}",
                 fontsize=14, fontweight="bold")

    for i, jn in enumerate(junctions):
        mask   = preds_df["junction"] == jn
        jdf    = preds_df[mask].reset_index(drop=True)
        y_true = jdf[TARGET].values
        y_pred = jdf[col].dropna().values
        n = min(len(y_true), len(y_pred))
        y_true, y_pred = y_true[:n], y_pred[:n]
        residuals = y_true - y_pred

        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)

        # Time-series overlay
        show = min(300, n)
        axes[i][0].plot(range(show), y_true[:show],
                        label="Actual", color="steelblue", lw=1.2)
        axes[i][0].plot(range(show), y_pred[:show],
                        label="Predicted", color="tomato", lw=1.2, alpha=0.85)
        axes[i][0].set_title(f"Jn {jn} — Actual vs Predicted  "
                              f"MAE={mae:.3f}  R²={r2:.4f}")
        axes[i][0].set_xlabel("Row index (val period)")
        axes[i][0].set_ylabel("Vehicles")
        axes[i][0].legend(fontsize=8)
        axes[i][0].grid(alpha=0.3)

        # Scatter
        axes[i][1].scatter(y_true, y_pred, alpha=0.25, s=8, color="purple")
        lim = [min(y_true.min(), y_pred.min()),
               max(y_true.max(), y_pred.max())]
        axes[i][1].plot(lim, lim, "r--", lw=1.5, label="Perfect fit")
        axes[i][1].set_xlabel("Actual")
        axes[i][1].set_ylabel("Predicted")
        axes[i][1].set_title(f"Jn {jn} — Scatter")
        axes[i][1].legend(fontsize=8)
        axes[i][1].grid(alpha=0.3)

        # Residuals histogram
        axes[i][2].hist(residuals, bins=50, color="mediumseagreen",
                        edgecolor="white", alpha=0.85)
        axes[i][2].axvline(0, color="red", linestyle="--", lw=1.5)
        axes[i][2].axvline(np.mean(residuals), color="navy", lw=1.2,
                           label=f"Mean={np.mean(residuals):.2f}")
        axes[i][2].set_title(f"Jn {jn} — Residuals")
        axes[i][2].set_xlabel("Residual")
        axes[i][2].legend(fontsize=8)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, f"eval_{mname.lower()}_per_junction.png")
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  ✔ eval_{mname.lower()}_per_junction.png saved")

# ── Cross-model MAE bar chart per junction ────────────────
n_jn = len(junctions)
fig, axes = plt.subplots(1, n_jn, figsize=(6 * n_jn, 5), sharey=False)
if n_jn == 1:
    axes = [axes]
fig.suptitle("Model MAE Comparison — per junction",
             fontsize=13, fontweight="bold")
colors = ["#2196F3","#4CAF50","#FF9800","#9C27B0"]

for ax, jn in zip(axes, junctions):
    jn_df = metrics_df[metrics_df["Junction"] == jn].sort_values("MAE")
    bars  = ax.bar(jn_df["Model"], jn_df["MAE"],
                   color=colors[:len(jn_df)], edgecolor="white", width=0.5)
    ax.set_title(f"Junction {jn}")
    ax.set_ylabel("MAE")
    ax.tick_params(axis="x", rotation=20)
    for bar, val in zip(bars, jn_df["MAE"]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01 * max(jn_df["MAE"]),
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
path = os.path.join(PLOT_DIR, "model_comparison_per_junction.png")
fig.savefig(path, dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✔ model_comparison_per_junction.png saved")

# ════════════════════════════════════════════════════════
#  3C.  TIME-SERIES CROSS-VALIDATION — PER JUNCTION
#
#  For each junction:
#    Combine that junction's train + val rows (time-ordered)
#    Run 5-fold TimeSeriesSplit on those rows only
#    Fit/score each tree model on each fold
# ════════════════════════════════════════════════════════
print("\n── 3C. Per-Junction Time-Series Cross-Validation ────")

# Load tree models (global versions)
models_cv = {}
for mname, fname in [("XGBoost",     "xgboost_global.pkl"),
                      ("LightGBM",    "lightgbm_global.pkl"),
                      ("RandomForest","randomforest_global.pkl")]:
    pkl = os.path.join(MODEL_DIR, fname)
    if os.path.exists(pkl):
        models_cv[mname] = joblib.load(pkl)
    else:
        # fallback: try per-junction naming for junction 1
        pkl2 = os.path.join(MODEL_DIR, f"{mname.lower()}_junction{junctions[0]}.pkl")
        if os.path.exists(pkl2):
            models_cv[mname] = joblib.load(pkl2)

if not models_cv:
    print("  ⚠  No serialised models found — skipping CV.")
else:
    tscv      = TimeSeriesSplit(n_splits=5)
    cv_records = []

    for jn in junctions:
        print(f"\n  Junction {jn}")
        # Combine in time order
        jn_all = pd.concat([
            train_df[train_df["junction"] == jn],
            val_df[val_df["junction"]     == jn]
        ]).sort_values(["year","month","hour"]).reset_index(drop=True)

        X_jn = jn_all[FEATURE_COLS].values
        y_jn = jn_all[TARGET].values

        for mname, model in models_cv.items():
            fold_maes, fold_rmses, fold_r2s = [], [], []
            for fold, (tr_i, vl_i) in enumerate(tscv.split(X_jn)):
                model.fit(X_jn[tr_i], y_jn[tr_i])
                p = model.predict(X_jn[vl_i])
                fold_maes.append(mean_absolute_error(y_jn[vl_i], p))
                fold_rmses.append(np.sqrt(mean_squared_error(y_jn[vl_i], p)))
                fold_r2s.append(r2_score(y_jn[vl_i], p))

            rec = {
                "Junction" : jn,
                "Model"    : mname,
                "MAE_mean" : round(np.mean(fold_maes),  4),
                "MAE_std"  : round(np.std(fold_maes),   4),
                "RMSE_mean": round(np.mean(fold_rmses), 4),
                "R2_mean"  : round(np.mean(fold_r2s),   4),
            }
            cv_records.append(rec)
            print(f"    {mname:<15} MAE={rec['MAE_mean']} "
                  f"± {rec['MAE_std']}  R²={rec['R2_mean']}")

    cv_df = pd.DataFrame(cv_records)
    cv_df.to_csv(os.path.join(MODEL_DIR, "cv_summary.csv"), index=False)
    print(f"\n✔ cv_summary.csv saved.")

    # ── CV boxplot per junction ──────────────────────────
    # (re-collect fold-level MAEs for plotting)
    cv_fold_data = {jn: {m: [] for m in models_cv} for jn in junctions}
    for jn in junctions:
        jn_all = pd.concat([
            train_df[train_df["junction"] == jn],
            val_df[val_df["junction"]     == jn]
        ]).sort_values(["year","month","hour"]).reset_index(drop=True)
        X_jn = jn_all[FEATURE_COLS].values
        y_jn = jn_all[TARGET].values
        for mname, model in models_cv.items():
            for _, (tr_i, vl_i) in enumerate(tscv.split(X_jn)):
                model.fit(X_jn[tr_i], y_jn[tr_i])
                p = model.predict(X_jn[vl_i])
                cv_fold_data[jn][mname].append(
                    mean_absolute_error(y_jn[vl_i], p))

    fig, axes = plt.subplots(1, len(junctions),
                             figsize=(6 * len(junctions), 5))
    if len(junctions) == 1:
        axes = [axes]
    fig.suptitle("5-Fold CV MAE Distribution — per junction",
                 fontsize=13, fontweight="bold")
    bp_colors = ["#2196F3","#4CAF50","#FF9800"]

    for ax, jn in zip(axes, junctions):
        data   = [cv_fold_data[jn][m] for m in models_cv]
        labels = list(models_cv.keys())
        bp = ax.boxplot(data, labels=labels, patch_artist=True,
                        medianprops={"color":"red","linewidth":2})
        for patch, c in zip(bp["boxes"], bp_colors):
            patch.set_facecolor(c); patch.set_alpha(0.6)
        ax.set_title(f"Junction {jn}")
        ax.set_ylabel("MAE")
        ax.tick_params(axis="x", rotation=15)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, "cv_boxplot_per_junction.png")
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"✔ cv_boxplot_per_junction.png saved.")

    # ── Bias-variance diagnosis per junction ─────────────
    print("\n── Bias-Variance Diagnosis (per junction) ─────────")
    for rec in cv_records:
        cv_ratio = rec["MAE_std"] / max(rec["MAE_mean"], 1e-9)
        if rec["MAE_mean"] > 8:
            verdict = "⚠  HIGH BIAS"
        elif cv_ratio > 0.3:
            verdict = "⚠  HIGH VARIANCE"
        else:
            verdict = "✅  Good fit"
        print(f"  Jn {rec['Junction']}  {rec['Model']:<15} "
              f"CV-ratio={cv_ratio:.3f}  {verdict}")

print(f"\n  Plots → {PLOT_DIR}")
print("\n✅ Component 3 complete.")

Models    : ['XGBoost', 'LightGBM', 'RandomForest']
Junctions : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

── 3A. Per-Junction Validation Metrics ─────────────

  Model : XGBoost
    Jn 1 → MAE=4.7145  RMSE=6.4318  R²=0.9277  MAPE=7.4488%
    Jn 2 → MAE=2.6755  RMSE=3.4455  R²=0.7207  MAPE=14.4064%
    Jn 3 → MAE=4.3623  RMSE=5.5131  R²=0.4928  MAPE=34.1262%
    Jn 4 → MAE=2.0666  RMSE=2.6107  R²=0.3557  MAPE=35.6666%

  Model : LightGBM
    Jn 1 → MAE=4.3948  RMSE=6.1314  R²=0.9343  MAPE=6.8472%
    Jn 2 → MAE=2.5038  RMSE=3.1945  R²=0.7599  MAPE=13.7477%
    Jn 3 → MAE=4.3404  RMSE=5.5399  R²=0.4879  MAPE=33.5212%
    Jn 4 → MAE=2.0756  RMSE=2.6224  R²=0.3499  MAPE=35.9859%

  Model : RandomForest
    Jn 1 → MAE=5.0439  RMSE=6.9672  R²=0.9152  MAPE=7.3756%
    Jn 2 → MAE=2.7402  RMSE=3.2947  R²=0.7446  MAPE=13.9094%
    Jn 3 → MAE=4.4861  RMSE=5.696  R²=0.4586  MAPE=34.9699%
    Jn 4 → MAE=1.8519  RMSE=2.3023  R²=0.499  MAPE=35.9094%

✔ evaluation_metrics.csv saved.

── Ov

In [7]:
# ═══════════════════════════════════════════════════════════
#  COMPONENT 4 — REFINEMENT  (per-junction)
#  Fine-tunes LightGBM per junction; compares before/after.
# ═══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os, warnings, joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, learning_curve
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

SAVE_DIR  = r"C:\Users\syin\Downloads\Uber_project\Datasets"
MODEL_DIR = os.path.join(SAVE_DIR, "models")
PLOT_DIR  = os.path.join(SAVE_DIR, "evaluation_plots")
os.makedirs(PLOT_DIR, exist_ok=True)

TARGET = "Vehicles"
train_df = pd.read_csv(os.path.join(SAVE_DIR, "train_split.csv"))
val_df   = pd.read_csv(os.path.join(SAVE_DIR, "val_split.csv"))
preds_df = pd.read_csv(os.path.join(MODEL_DIR, "val_predictions.csv"))
FEATURE_COLS = pd.read_csv(
    os.path.join(SAVE_DIR, "feature_cols.csv"))["feature"].tolist()
junctions = sorted(train_df["junction"].unique())
tscv      = TimeSeriesSplit(n_splits=5)

def metrics(y, p):
    mae  = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    r2   = r2_score(y, p)
    mape = np.mean(np.abs((y - p) / np.where(y == 0, 1, y))) * 100
    return mae, rmse, r2, mape

# ── Load baseline global LightGBM ────────────────────────
baseline_path = os.path.join(MODEL_DIR, "lightgbm_global.pkl")
if not os.path.exists(baseline_path):
    # fallback if per-junction models were trained
    baseline_path = os.path.join(
        MODEL_DIR, f"lightgbm_junction{junctions[0]}.pkl")
baseline_model = joblib.load(baseline_path)
print(f"✔ Loaded baseline : {os.path.basename(baseline_path)}\n")

# ════════════════════════════════════════════════════════
#  4A.  PER-JUNCTION ERROR ANALYSIS
# ════════════════════════════════════════════════════════
print("── 4A. Per-Junction Error Analysis ────────────────")

fig, axes = plt.subplots(len(junctions), 3,
                          figsize=(18, 5 * len(junctions)))
if len(junctions) == 1:
    axes = [axes]
fig.suptitle("Error Analysis per Junction — LightGBM (baseline)",
             fontsize=13, fontweight="bold")

error_summary = []
for i, jn in enumerate(junctions):
    mask   = val_df["junction"] == jn
    X_vj   = val_df.loc[mask, FEATURE_COLS].values
    y_vj   = val_df.loc[mask, TARGET].values
    pred_j = baseline_model.predict(X_vj)
    res_j  = y_vj - pred_j
    abs_j  = np.abs(res_j)

    mae_b, rmse_b, r2_b, mape_b = metrics(y_vj, pred_j)
    error_summary.append({
        "junction": jn, "MAE_before": round(mae_b,4),
        "RMSE_before": round(rmse_b,4), "R2_before": round(r2_b,4)
    })
    print(f"  Junction {jn}: MAE={mae_b:.4f}  RMSE={rmse_b:.4f}  R²={r2_b:.4f}")

    # Error by hour
    tmp = val_df[mask].copy()
    tmp["abs_error"] = abs_j
    hourly = tmp.groupby("hour")["abs_error"].mean()
    axes[i][0].bar(hourly.index, hourly.values, color="steelblue",
                   edgecolor="white")
    axes[i][0].set_title(f"Jn {jn} — MAE by hour")
    axes[i][0].set_xlabel("Hour"); axes[i][0].set_ylabel("MAE")
    axes[i][0].grid(alpha=0.3)

    # Actual vs error scatter
    axes[i][1].scatter(y_vj, abs_j, alpha=0.2, s=6, color="purple")
    axes[i][1].set_title(f"Jn {jn} — Actual vs Abs Error")
    axes[i][1].set_xlabel("Actual Vehicles")
    axes[i][1].set_ylabel("Absolute Error")
    axes[i][1].grid(alpha=0.3)

    # Residual distribution
    axes[i][2].hist(res_j, bins=50, color="mediumseagreen",
                    edgecolor="white", alpha=0.85)
    axes[i][2].axvline(0, color="red", linestyle="--", lw=1.5)
    axes[i][2].axvline(np.mean(res_j), color="navy", lw=1.2,
                       label=f"Mean={np.mean(res_j):.2f}")
    axes[i][2].set_title(f"Jn {jn} — Residuals")
    axes[i][2].legend(fontsize=8)

plt.tight_layout()
path = os.path.join(PLOT_DIR, "error_analysis_per_junction.png")
fig.savefig(path, dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✔ error_analysis_per_junction.png saved")

# ════════════════════════════════════════════════════════
#  4B.  FEATURE IMPORTANCE (global baseline)
# ════════════════════════════════════════════════════════
print("\n── 4B. Feature Importance ──────────────────────────")
X_tr_all = train_df[FEATURE_COLS].values
y_tr_all = train_df[TARGET].values
importances = baseline_model.feature_importances_
feat_df = pd.DataFrame({"feature": FEATURE_COLS,
                         "importance": importances}
                        ).sort_values("importance", ascending=False)
feat_df.to_csv(os.path.join(MODEL_DIR, "feature_importance.csv"), index=False)
print(f"  Top 10: {feat_df.head(10)['feature'].tolist()}")

total_imp  = feat_df["importance"].sum()
threshold  = total_imp * 0.005
good_feats = feat_df[feat_df["importance"] >= threshold]["feature"].tolist()
low_feats  = feat_df[feat_df["importance"] <  threshold]["feature"].tolist()
print(f"  Removed low-importance ({len(low_feats)}): {low_feats}")
print(f"  Retained : {len(good_feats)} features")

# ════════════════════════════════════════════════════════
#  4C.  FINE-TUNE LightGBM PER JUNCTION
#        Uses refined feature set + narrow GridSearch
# ════════════════════════════════════════════════════════
print("\n── 4C. Fine-Tuning LightGBM per Junction ───────────")

fine_grid = {
    "n_estimators"      : [500, 600, 700],
    "learning_rate"     : [0.03, 0.05, 0.07],
    "num_leaves"        : [47, 63, 79],
    "max_depth"         : [5, 6, 7],
    "min_child_samples" : [15, 20, 30],
    "reg_alpha"         : [0, 0.05, 0.1],
    "reg_lambda"        : [0, 0.1, 0.5],
}

refinement_rows = []
final_preds_df  = val_df[["junction","year","month","hour",TARGET]].copy()
final_preds_df["pred_before"] = np.nan
final_preds_df["pred_after"]  = np.nan

for jn in junctions:
    print(f"\n  Junction {jn}")
    tr_jn   = train_df[train_df["junction"] == jn]
    vl_jn   = val_df[val_df["junction"]     == jn]
    X_tr    = tr_jn[good_feats].values
    y_tr    = tr_jn[TARGET].values
    X_vl    = vl_jn[good_feats].values
    y_vl    = vl_jn[TARGET].values

    if len(y_vl) == 0:
        print("    ⚠ No val rows — skipping.")
        continue

    # Baseline on this junction (using refined features)
    pred_before = baseline_model.predict(
        vl_jn[FEATURE_COLS].values)   # baseline uses original features
    mae_b, rmse_b, r2_b, _ = metrics(y_vl, pred_before)

    # GridSearch fine-tune
    gs = GridSearchCV(
        lgb.LGBMRegressor(subsample=1.0, colsample_bytree=1.0,
                          random_state=42, n_jobs=-1, verbose=-1),
        fine_grid, cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1, verbose=0, refit=True
    )
    gs.fit(X_tr, y_tr)
    final_model = gs.best_estimator_

    pred_after = final_model.predict(X_vl)
    mae_a, rmse_a, r2_a, mape_a = metrics(y_vl, pred_after)

    print(f"    Before : MAE={mae_b:.4f}  RMSE={rmse_b:.4f}  R²={r2_b:.4f}")
    print(f"    After  : MAE={mae_a:.4f}  RMSE={rmse_a:.4f}  R²={r2_a:.4f}  "
          f"{'✅' if mae_a < mae_b else '❌'}")
    print(f"    Params : {gs.best_params_}")

    joblib.dump(final_model,
                os.path.join(MODEL_DIR,
                             f"FINAL_lgb_junction{jn}.pkl"))

    final_preds_df.loc[vl_jn.index, "pred_before"] = pred_before
    final_preds_df.loc[vl_jn.index, "pred_after"]  = pred_after
    refinement_rows.append({
        "Junction": jn,
        "MAE_before": round(mae_b,4), "RMSE_before": round(rmse_b,4),
        "R2_before": round(r2_b,4),
        "MAE_after": round(mae_a,4), "RMSE_after": round(rmse_a,4),
        "R2_after": round(r2_a,4), "MAPE_after": round(mape_a,4)
    })

refine_df = pd.DataFrame(refinement_rows)
refine_df.to_csv(os.path.join(MODEL_DIR, "refinement_summary.csv"),
                 index=False)
final_preds_df.to_csv(os.path.join(MODEL_DIR, "final_predictions.csv"),
                      index=False)

# ════════════════════════════════════════════════════════
#  4D.  BEFORE vs AFTER PLOTS — per junction
# ════════════════════════════════════════════════════════
print("\n── 4D. Before vs After plots ───────────────────────")

fig, axes = plt.subplots(len(junctions), 2,
                          figsize=(18, 5 * len(junctions)))
if len(junctions) == 1:
    axes = [axes]
fig.suptitle("Before vs After Refinement — per Junction",
             fontsize=13, fontweight="bold")

for i, row in refine_df.iterrows():
    jn   = row["Junction"]
    mask = final_preds_df["junction"] == jn
    fdf  = final_preds_df[mask].reset_index(drop=True)
    y    = fdf[TARGET].values
    pb   = fdf["pred_before"].values
    pa   = fdf["pred_after"].values
    n    = min(300, len(y))

    axes[i][0].plot(range(n), y[:n],  label="Actual",
                    color="steelblue", lw=1.2)
    axes[i][0].plot(range(n), pb[:n], label=f"Before MAE={row['MAE_before']}",
                    color="tomato", lw=1.2, alpha=0.85)
    axes[i][0].set_title(f"Jn {jn} — Before refinement")
    axes[i][0].set_ylabel("Vehicles"); axes[i][0].legend(fontsize=8)
    axes[i][0].grid(alpha=0.3)

    axes[i][1].plot(range(n), y[:n],  label="Actual",
                    color="steelblue", lw=1.2)
    axes[i][1].plot(range(n), pa[:n], label=f"After  MAE={row['MAE_after']}",
                    color="mediumseagreen", lw=1.2, alpha=0.85)
    axes[i][1].set_title(f"Jn {jn} — After refinement")
    axes[i][1].set_ylabel("Vehicles"); axes[i][1].legend(fontsize=8)
    axes[i][1].grid(alpha=0.3)

plt.tight_layout()
path = os.path.join(PLOT_DIR, "before_vs_after_per_junction.png")
fig.savefig(path, dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✔ before_vs_after_per_junction.png saved")

# ── Final summary ────────────────────────────────────────
print("\n╔══════════════════════════════════════════════════════╗")
print("║         FINAL REFINEMENT SUMMARY (per junction)     ║")
print("╠══════════════════════════════════════════════════════╣")
for _, row in refine_df.iterrows():
    delta = row["MAE_after"] - row["MAE_before"]
    status = "✅ improved" if delta < 0 else "❌ worse"
    print(f"║  Jn {row['Junction']}  MAE {row['MAE_before']:.4f} → "
          f"{row['MAE_after']:.4f}  ({delta:+.4f})  {status:<12}  "
          f"R²={row['R2_after']:.4f}  ║")
print("╚══════════════════════════════════════════════════════╝")

print("\n✔ FINAL_lgb_junction<N>.pkl saved per junction")
print("✔ refinement_summary.csv saved")
print("✔ final_predictions.csv saved")
print("\n✅ Component 4 complete.")

✔ Loaded baseline : lightgbm_global.pkl

── 4A. Per-Junction Error Analysis ────────────────
  Junction 1: MAE=4.3948  RMSE=6.1314  R²=0.9343
  Junction 2: MAE=2.5038  RMSE=3.1945  R²=0.7599
  Junction 3: MAE=4.3404  RMSE=5.5399  R²=0.4879
  Junction 4: MAE=2.0756  RMSE=2.6224  R²=0.3499
  ✔ error_analysis_per_junction.png saved

── 4B. Feature Importance ──────────────────────────
  Top 10: ['junction', 'temp_norm', 'rolling_std_24h', 'hour', 'rolling_std_6h', 'rolling_mean_6h', 'ewma_12h', 'pres', 'rolling_mean_24h', 'rolling_max_6h']
  Removed low-importance (6): ['time_of_day_enc', 'is_workday', 'is_holiday', 'is_special_event', 'is_rush_hour', 'is_weekend']
  Retained : 34 features

── 4C. Fine-Tuning LightGBM per Junction ───────────

  Junction 1
    Before : MAE=4.3948  RMSE=6.1314  R²=0.9343
    After  : MAE=4.1952  RMSE=5.9940  R²=0.9372  ✅
    Params : {'learning_rate': 0.03, 'max_depth': 5, 'min_child_samples': 15, 'n_estimators': 500, 'num_leaves': 47, 'reg_alpha': 0.1, 'r